# N-CMAPSS Windowing — Phase 4 (Fixed-Length Windows for MA1DCNN)

Turns preprocessing.ipynb's normalized, split, **row-level** output into
the fixed-length `(channels, L)` windows `MA1DCNN.forward()` actually
requires — the gap flagged in both `ma1dcnn.ipynb` ("a windowing
dataset—genuinely new code") and `pipeline_sequence_and_dependencies.docx`.

## Build scope for this notebook (agreed decision)

Same discipline as `loaders.ipynb` and `ma1dcnn.ipynb`: this notebook
builds and fully tests the windowing logic against small, synthetic,
in-memory "processed file" dicts — no Drive, no real N-CMAPSS data
needed to run or understand any cell below. Section 9 ("Run it") is the
only cell that needs Drive and real `data/preprocessed/*.npz` files;
every other cell, including the self-check, runs standalone.

## Confirmed input (resolved since this notebook was first written)

An earlier version of this notebook flagged `preprocessing.ipynb`'s real
load/save behavior as an open, unconfirmed question and pointed this
notebook's input at `data/processed/` (`loaders.ipynb`'s own output) as
a placeholder. `preprocessing.ipynb`'s full code has since been reviewed
directly — see `pipeline_sequence_and_dependencies.docx`, Section 3 —
and that question is now resolved: `preprocessing.ipynb` reads
`loaders.ipynb`'s saved `data/processed/*.npz` files, adds its own
unit-based train/validation split, z-score normalization (training-only
statistics), and training-only noise augmentation, and saves two files
per source file — `<filename>_train.npz` and `<filename>_val.npz` — to
a separate `data/preprocessed/` folder. `A`, `Y`, and `labels` pass
through that notebook completely unchanged, so this notebook's own
unit/cycle grouping and labeling logic below needs no changes — only
the input folder and split names below were updated to match.

## Four judgment calls this notebook makes explicit

None of these four are specified anywhere reviewed in Akindoju (2025)
so far — each is a named, reversible decision, not a silent default:

1. **Window length and stride** — how many consecutive cycles make one
   window, and how far the window slides between samples.
2. **Label collapsing rule** — a window covers many rows, each with its
   own 5-flag label; this decides how those become ONE label per window.
3. **Whether `W` (flight condition) becomes extra input channels** —
   the open question `ma1dcnn.ipynb`'s own `DEFAULT_N_INPUT_CHANNELS`
   comment flagged rather than resolved.
4. **What happens to a unit with fewer rows than one window** — skipped
   entirely, loudly, never silently padded with made-up data.

## What every section below does

1. **Constants** — the four judgment calls above, named once.
2. **`_collapse_window_labels`** — turns many rows' labels into one.
3. **`_windows_for_one_unit`** — slices one unit's already-ordered rows.
4. **`_stack_channels`** — builds the 28- or 32-channel input matrix.
5. **`_build_windows_for_file`** — groups by unit, orders by cycle,
   windows each unit separately, never crossing a unit boundary.
6. **Save/load helpers** — the same chunked, one-file-at-a-time,
   never-silent pattern `loaders.ipynb` already established.
7. **Tests and a self-check** — proving every claim above in code, plus
   one illustrative, non-test printout built specifically so the
   difference between the two label rules can be seen, not just read.


## 0. Imports

Only `os` and `numpy` — windowing works entirely on already-loaded
arrays, so it needs no `h5py` (that was loaders.ipynb's job alone).

In [2]:
import os
import tempfile

import numpy as np

## 1. Constants — every judgment call, named once

`FAULT_COLUMNS` and `NCMAPSS_FILES` are repeated here rather than
imported — the same standalone convention `loaders.ipynb` uses — so
this notebook can run and be understood with no other file open.

In [3]:
# Repeated from loaders.ipynb (not imported) so this notebook stays
# standalone -- column order must match exactly, since it decides which
# label column a fault flag lands in downstream.
FAULT_COLUMNS = ["fan_fail", "lpc_fail", "hpc_fail", "hpt_fail", "lpt_fail"]

# Repeated from loaders.ipynb -- same 9 files, same fixed order.
NCMAPSS_FILES = [
    "N-CMAPSS_DS01-005.h5", "N-CMAPSS_DS02-006.h5", "N-CMAPSS_DS03-012.h5",
    "N-CMAPSS_DS04.h5", "N-CMAPSS_DS05.h5", "N-CMAPSS_DS06.h5",
    "N-CMAPSS_DS07.h5", "N-CMAPSS_DS08a-009.h5", "N-CMAPSS_DS08c-008.h5",
]

# Where this notebook's INPUT lives -- preprocessing.ipynb's own save
# location, confirmed directly from its own DEFAULT_PREPROCESSED_DIR
# constant and code (see the "Confirmed input" note above). Renamed from
# an earlier DEFAULT_PROCESSED_DIR, which pointed at loaders.ipynb's
# output instead, back when preprocessing.ipynb's behavior was still
# unconfirmed.
DEFAULT_PREPROCESSED_DIR = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/preprocessed"

# Where this notebook's OUTPUT goes -- a NEW directory, deliberately not
# data/preprocessed/, so a per-row file is never confused with a windowed one.
DEFAULT_WINDOWED_DIR = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/windowed"

# Judgment call 1 of 4: how many consecutive cycles make one window, and
# how far apart consecutive windows start. Not one of Akindoju's tuned
# numbers -- picked to match the L=50 used throughout ma1dcnn.ipynb's own
# tests. stride == window_length means NON-overlapping windows: simplest
# to reason about and to test, and keeps windows statistically
# independent of each other. A smaller stride (overlapping windows, more
# training data, correlated samples) is a legitimate later tuning choice
# -- the code below supports it, this default just doesn't use it yet.
WINDOW_LENGTH = 50
WINDOW_STRIDE = 50

# Judgment call 2 of 4: how a window's many per-row labels become ONE
# label. "last" = the window's own final row's label (the window is read
# as "here is the history LEADING UP TO this point" -- standard framing
# for RUL-style prognostics). "any" = flagged if ANY row inside the
# window was a failure row, even if the window's last row is healthy.
# Both are implemented; see Section 11 for a worked example where they
# disagree.
LABEL_RULE = "last"

# Judgment call 3 of 4: whether W's 4 flight-condition columns become
# extra input channels. False keeps 28 channels, matching
# DEFAULT_N_INPUT_CHANNELS already baked into ma1dcnn.ipynb's
# BEFORE_/AFTER_TUNING_CONFIG. True gives 32 -- implemented so this
# question can be revisited later without rewriting the windowing logic.
INCLUDE_FLIGHT_CONDITIONS = False

# Which columns of A hold the unit id and the cycle number -- per
# loaders.ipynb's own comment: "A_dev # unit, cycle, Fc, hs". Untouched
# by preprocessing.ipynb (A passes through it unchanged), so these column
# indices are still correct for preprocessing.ipynb's saved output too.
UNIT_COL = 0
CYCLE_COL = 1


## 2. Collapse a window's per-row labels into one label

Every window covers `window_length` rows, and every row already has its
own 5-flag label from `loaders.ipynb`. This is Judgment call 2: how
those many labels become the ONE label training will actually use.

In [4]:
def _collapse_window_labels(window_labels, label_rule):
    """Turn one window's (window_length, 5) per-row labels into a single (5,) label.

    Parameters
    ----------
    window_labels : np.ndarray, shape (window_length, 5)
        The per-row label matrix for exactly the rows inside one window.
    label_rule : str
        "last" -- use the window's own final row's label.
        "any"  -- flag a column if ANY row inside the window set it.
    """
    if label_rule == "last":
        return window_labels[-1]  # the row the window "ends on" -- what happens right after this window's history
    elif label_rule == "any":
        # sum > 0 across rows -- catches a fault flagged ANYWHERE inside the window, not just at the end
        return (window_labels.sum(axis=0) > 0).astype(np.int8)
    else:
        # fail loudly on a typo'd rule name rather than silently defaulting to one of the two
        raise ValueError(f"label_rule must be 'last' or 'any', got {label_rule!r}")

## 3. Slice one already-ordered unit's rows into fixed-length windows

Takes rows that ALREADY belong to a single unit and are ALREADY sorted
by cycle (Section 5 handles that grouping and ordering) and slides a
`window_length`-wide window across them with step `stride`.

In [5]:
def _windows_for_one_unit(X, labels, window_length=WINDOW_LENGTH, stride=WINDOW_STRIDE, label_rule=LABEL_RULE):
    """Slice one unit's (n_rows, C) sensor matrix into (n_windows, C, window_length) windows.

    Parameters
    ----------
    X : np.ndarray, shape (n_rows, C)
        One unit's stacked sensor channels, already time-ordered.
    labels : np.ndarray, shape (n_rows, 5)
        That same unit's per-row labels, row-aligned with X.

    Returns
    -------
    windows : np.ndarray, shape (n_windows, C, window_length)
        Channels-first, matching what MA1DCNN.forward() expects directly.
    window_labels : np.ndarray, shape (n_windows, 5)
    starts : np.ndarray, shape (n_windows,)
        The row index (within this unit) each window started at -- kept
        for traceability, not used in training itself.
    """
    n_rows = X.shape[0]

    if n_rows < window_length:
        # Judgment call 4: a unit shorter than one window produces ZERO
        # windows -- never padded with made-up data. The caller (Section
        # 5) is responsible for printing that this happened, out loud.
        return (np.empty((0, X.shape[1], window_length), dtype=X.dtype),
                np.empty((0, labels.shape[1]), dtype=labels.dtype),
                np.empty((0,), dtype=np.int64))

    # Every valid starting row for a full-length window: the last
    # possible start is n_rows - window_length, so a window never reads
    # past this unit's own last row.
    starts = np.arange(0, n_rows - window_length + 1, stride)
    n_windows = len(starts)

    windows = np.empty((n_windows, X.shape[1], window_length), dtype=X.dtype)
    window_labels = np.empty((n_windows, labels.shape[1]), dtype=labels.dtype)

    for i, start in enumerate(starts):
        end = start + window_length
        windows[i] = X[start:end, :].T  # (window_length, C) -> (C, window_length): channels-first for the model
        window_labels[i] = _collapse_window_labels(labels[start:end, :], label_rule)

    return windows, window_labels, starts

In [5]:
def _test_windows_for_one_unit_shapes_and_boundaries():
    n_rows, n_channels, L, stride = 7, 2, 3, 2
    X = np.arange(n_rows * n_channels, dtype=np.float64).reshape(n_rows, n_channels)
    labels = np.zeros((n_rows, 5), dtype=np.int8)
    labels[5, 2] = 1  # a fault flag on row index 5 (the 6th row) only

    windows, window_labels, starts = _windows_for_one_unit(X, labels, window_length=L, stride=stride, label_rule="last")

    assert starts.tolist() == [0, 2, 4], starts.tolist()
    assert windows.shape == (3, n_channels, L), windows.shape
    assert window_labels.shape == (3, 5), window_labels.shape

    expected_window1 = X[2:5, :].T
    assert np.array_equal(windows[1], expected_window1)

    # Row index 5 is the only row with a fault flag. Window 0 = rows
    # [0,1,2], window 1 = rows [2,3,4], window 2 = rows [4,5,6]. Only
    # window 2's SPAN includes row 5 -- but row 5 is NOT window 2's last
    # row (row 6 is). This is exactly where "last" and "any" disagree.
    assert window_labels[0].tolist() == [0, 0, 0, 0, 0]
    assert window_labels[1].tolist() == [0, 0, 0, 0, 0]
    assert window_labels[2].tolist() == [0, 0, 0, 0, 0], "label_rule='last' looks at row 6, which has no fault"

    windows_any, window_labels_any, _ = _windows_for_one_unit(X, labels, window_length=L, stride=stride, label_rule="any")
    assert window_labels_any[2].tolist() == [0, 0, 1, 0, 0], "label_rule='any' should catch row 5's flag inside window 2"

    # A unit shorter than window_length must produce ZERO windows, not an error and not a padded fake window.
    windows_short, labels_short, starts_short = _windows_for_one_unit(
        X[:2, :], labels[:2, :], window_length=L, stride=stride, label_rule="last")
    assert windows_short.shape == (0, n_channels, L)
    assert starts_short.shape == (0,)

    print("_windows_for_one_unit: PASSED (window count, shapes, content, both label rules, short-unit skip)")


_test_windows_for_one_unit_shapes_and_boundaries()

_windows_for_one_unit: PASSED (window count, shapes, content, both label rules, short-unit skip)


## 4. Stack the model's input channels

Judgment call 3: whether `W` becomes extra channels. `X_s` and `X_v`
are ALWAYS included (28 channels); `W` is concatenated only when
`INCLUDE_FLIGHT_CONDITIONS=True` (32 channels), matching the open
question flagged in `ma1dcnn.ipynb`'s own constants section.

In [6]:
def _stack_channels(file_data, include_flight_conditions=INCLUDE_FLIGHT_CONDITIONS):
    """Build the (n_rows, C) channel matrix windowing will slice up.

    C is 28 (X_s + X_v only) by default, or 32 if W is included -- see
    Judgment call 3 above. Column order is fixed: [W if included] then
    X_s then X_v, so channel index is reproducible run to run.
    """
    if include_flight_conditions:
        return np.concatenate([file_data["W"], file_data["X_s"], file_data["X_v"]], axis=1)
    return np.concatenate([file_data["X_s"], file_data["X_v"]], axis=1)

## 5. Group one file's rows by unit, order by cycle, window each unit separately

This is the function that actually answers "how does windowing know
where one engine ends and the next begins": it uses `A`'s own unit and
cycle columns — already saved by `loaders.ipynb` — rather than
inventing any new bookkeeping.

In [7]:
def _build_windows_for_file(file_data, window_length=WINDOW_LENGTH, stride=WINDOW_STRIDE,
                             label_rule=LABEL_RULE, include_flight_conditions=INCLUDE_FLIGHT_CONDITIONS):
    """Window one file's worth of rows, unit by unit, never crossing a unit boundary.

    Parameters
    ----------
    file_data : dict
        Exactly what load_processed_file() below returns -- keys W, X_s,
        X_v, A, Y, labels, all row-aligned, matching loaders.ipynb's own
        saved format.

    Returns
    -------
    windows : np.ndarray, shape (n_windows, C, window_length)
    window_labels : np.ndarray, shape (n_windows, 5)
    unit_ids : np.ndarray, shape (n_windows,) -- which unit each window came from, for traceability
    start_cycles : np.ndarray, shape (n_windows,) -- the cycle number each window started at
    """
    X_full = _stack_channels(file_data, include_flight_conditions)
    labels_full = file_data["labels"]
    A = file_data["A"]

    unit_col = A[:, UNIT_COL]
    cycle_col = A[:, CYCLE_COL]

    all_windows, all_labels, all_unit_ids, all_start_cycles = [], [], [], []
    skipped_units = []  # never silent -- collected here, printed once below

    for unit in np.unique(unit_col):
        unit_mask = unit_col == unit                                    # every row belonging to this ONE unit
        order = np.argsort(cycle_col[unit_mask], kind="stable")         # time-order WITHIN this unit only

        X_unit = X_full[unit_mask][order]
        labels_unit = labels_full[unit_mask][order]
        cycles_unit = cycle_col[unit_mask][order]

        windows, window_labels, starts = _windows_for_one_unit(
            X_unit, labels_unit, window_length=window_length, stride=stride, label_rule=label_rule)

        if windows.shape[0] == 0:
            skipped_units.append((unit, X_unit.shape[0]))
            continue  # this unit contributes NOTHING -- no partial or padded window

        all_windows.append(windows)
        all_labels.append(window_labels)
        all_unit_ids.append(np.full(windows.shape[0], unit, dtype=unit_col.dtype))
        all_start_cycles.append(cycles_unit[starts])  # convert row-index starts into actual cycle numbers

    if skipped_units:
        for unit, n in skipped_units:
            print(f"    [skip] unit {unit}: only {n} rows (< window_length={window_length}), no windows produced")

    if not all_windows:
        # every unit in this file was too short -- return well-shaped empty arrays, not None
        n_channels = X_full.shape[1]
        return (np.empty((0, n_channels, window_length), dtype=X_full.dtype),
                np.empty((0, labels_full.shape[1]), dtype=labels_full.dtype),
                np.empty((0,), dtype=unit_col.dtype),
                np.empty((0,), dtype=cycle_col.dtype))

    return (np.concatenate(all_windows, axis=0),
            np.concatenate(all_labels, axis=0),
            np.concatenate(all_unit_ids, axis=0),
            np.concatenate(all_start_cycles, axis=0))

In [8]:
def _test_build_windows_for_file_respects_unit_boundaries():
    rng = np.random.default_rng(0)

    # Row counts chosen so window_length=3, stride=3 divides each unit's
    # rows EXACTLY (9/3=3 windows, 6/3=2 windows) -- keeps the expected
    # window boundaries simple to reason about by hand.
    unit1_rows, unit2_rows = 9, 6
    n_rows = unit1_rows + unit2_rows

    A = np.zeros((n_rows, 4))
    A[:unit1_rows, 0] = 1
    A[:unit1_rows, 1] = np.arange(1, unit1_rows + 1)
    A[unit1_rows:, 0] = 2
    A[unit1_rows:, 1] = np.arange(1, unit2_rows + 1)

    X_s = rng.random((n_rows, 14))
    X_v = rng.random((n_rows, 14))
    W = rng.random((n_rows, 4))

    # Fault flag on the LAST cycle of each unit -- only the window whose
    # own last row is that final row should ever see it.
    labels = np.zeros((n_rows, 5), dtype=np.int8)
    labels[unit1_rows - 1, 0] = 1
    labels[unit1_rows + unit2_rows - 1, 3] = 1

    file_data = {"W": W, "X_s": X_s, "X_v": X_v, "A": A, "Y": np.zeros((n_rows, 1)), "labels": labels}

    windows, window_labels, unit_ids, start_cycles = _build_windows_for_file(
        file_data, window_length=3, stride=3, label_rule="last")

    # 9 rows / stride 3 = 3 windows for unit 1; 6 rows / stride 3 = 2 windows for unit 2 -- 5 total, in unit order.
    assert windows.shape == (5, 28, 3), windows.shape
    assert unit_ids.tolist() == [1, 1, 1, 2, 2], unit_ids.tolist()
    assert start_cycles.tolist() == [1, 4, 7, 1, 4], start_cycles.tolist()

    # Confirm each window's DATA really is the right slice of X_s -- not just the right shape.
    for i, uid in enumerate(unit_ids):
        offset = 0 if uid == 1 else unit1_rows
        start_row = offset + int(start_cycles[i] - 1)
        expected = X_s[start_row:start_row + 3, :]
        assert np.array_equal(windows[i, :14, :].T, expected), (i, uid)

    assert window_labels[2].tolist() == [1, 0, 0, 0, 0], "unit 1's last window should catch its end-of-life fan_fail"
    assert window_labels[4].tolist() == [0, 0, 0, 1, 0], "unit 2's last window should catch its end-of-life hpt_fail"
    for i in [0, 1, 3]:
        assert window_labels[i].tolist() == [0, 0, 0, 0, 0], i

    # include_flight_conditions=True should give 32 channels, same window count.
    windows32, _, _, _ = _build_windows_for_file(file_data, window_length=3, stride=3,
                                                  label_rule="last", include_flight_conditions=True)
    assert windows32.shape == (5, 32, 3), windows32.shape

    # A unit with fewer rows than window_length must be skipped entirely, never silently.
    file_data_tiny_unit = dict(file_data)
    A_tiny = A.copy()
    A_tiny[unit1_rows:, 0] = 3
    file_data_tiny_unit["A"] = A_tiny
    windows_tiny, _, unit_ids_tiny, _ = _build_windows_for_file(
        file_data_tiny_unit, window_length=7, stride=7, label_rule="last")
    assert set(unit_ids_tiny.tolist()) == {1}, unit_ids_tiny.tolist()

    print("_build_windows_for_file: PASSED (unit grouping, cycle ordering, boundary respect, both channel modes, short-unit skip)")


_test_build_windows_for_file_respects_unit_boundaries()

    [skip] unit 3.0: only 6 rows (< window_length=7), no windows produced
_build_windows_for_file: PASSED (unit grouping, cycle ordering, boundary respect, both channel modes, short-unit skip)


## 6. Read one already-processed file back

Same pattern as `loaders.ipynb`'s own `load_processed_file()` —
repeated here rather than imported, for the same standalone reason.

In [9]:
def load_processed_file(filename, data_dir=DEFAULT_PREPROCESSED_DIR, split="train"):
    """Load ONE already-preprocessed file back from its saved .npz.

    split is "train" or "val" (preprocessing.ipynb's own split names --
    there is no longer a single combined "dev" split to read, since
    preprocessing.ipynb's own train/validation split replaced it).
    """
    out_name = filename.replace(".h5", f"_{split}.npz")
    path = os.path.join(data_dir, out_name)
    with np.load(path) as npz:
        return {key: npz[key] for key in npz.files}


## 7. Process and save all files' windows — ONE FILE AT A TIME

The same chunked, never-silent discipline `loaders.ipynb`'s
`process_and_save_all_files()` already established: one file loaded,
windowed, saved, and freed before the next begins, with every step
printed as it happens.

In [10]:
def process_and_save_all_windowed_files(data_dir=DEFAULT_PREPROCESSED_DIR, out_dir=DEFAULT_WINDOWED_DIR,
                                         split="train", files=None, window_length=WINDOW_LENGTH,
                                         stride=WINDOW_STRIDE, label_rule=LABEL_RULE,
                                         include_flight_conditions=INCLUDE_FLIGHT_CONDITIONS):
    """Load, window, and save all files to disk ONE AT A TIME, for ONE split.

    Call this once with split="train" and again with split="val" --
    preprocessing.ipynb saves those as two separate files per source
    file, so there is no single combined split to window in one pass.

    Returns
    -------
    list[dict]
        One small summary dict per file -- {"filename", "n_rows",
        "n_windows", "output_path", "output_mb"} -- never the arrays
        themselves, so this function's return value stays small no
        matter how many files or windows are produced.
    """
    files = files if files is not None else NCMAPSS_FILES
    os.makedirs(out_dir, exist_ok=True)

    summaries = []
    running_total_windows = 0

    for i, filename in enumerate(files, start=1):
        print(f"[{i}/{len(files)}] Loading {filename} ({split}) ...")  # printed BEFORE the slow step, so a hang is visible

        file_data = load_processed_file(filename, data_dir=data_dir, split=split)  # only THIS file in memory now
        n_rows = file_data["W"].shape[0]
        print(f"    read {n_rows:,} rows")

        windows, window_labels, unit_ids, start_cycles = _build_windows_for_file(
            file_data, window_length=window_length, stride=stride,
            label_rule=label_rule, include_flight_conditions=include_flight_conditions)
        n_windows = windows.shape[0]
        print(f"    built {n_windows:,} windows (window_length={window_length}, stride={stride})")

        out_name = filename.replace(".h5", f"_{split}_windowed.npz")
        out_path = os.path.join(out_dir, out_name)
        np.savez_compressed(out_path, X=windows, y=window_labels, unit_id=unit_ids, start_cycle=start_cycles)
        output_mb = os.path.getsize(out_path) / (1024 * 1024)
        print(f"    saved to {out_path} ({output_mb:.1f} MB)")

        summaries.append({
            "filename": filename, "n_rows": n_rows, "n_windows": n_windows,
            "output_path": out_path, "output_mb": output_mb,
        })
        running_total_windows += n_windows

        del file_data, windows, window_labels, unit_ids, start_cycles  # free this file's memory before the next

    print(f"Done -- {len(files)} files processed, {running_total_windows:,} total windows saved to {out_dir}")
    return summaries


## 8. Read windowed files back

In [11]:
def load_windowed_file(filename, out_dir=DEFAULT_WINDOWED_DIR, split="train"):
    """Load ONE file's already-saved windows back from its .npz."""
    out_name = filename.replace(".h5", f"_{split}_windowed.npz")
    path = os.path.join(out_dir, out_name)
    with np.load(path) as npz:
        return {key: npz[key] for key in npz.files}


def iter_windowed_files(out_dir=DEFAULT_WINDOWED_DIR, split="train", files=None):
    """Yield one file's windows at a time -- the same chunked read pattern loaders.ipynb uses."""
    files = files if files is not None else NCMAPSS_FILES
    for filename in files:
        yield filename, load_windowed_file(filename, out_dir=out_dir, split=split)


## 9. Run it

Processes and saves windows for all 9 files from Drive, once per split.

**This cell needs Drive mounted and `data/preprocessed/*_train.npz` /
`*_val.npz` actually present** at `DEFAULT_PREPROCESSED_DIR` --
confirmed in `pipeline_sequence_and_dependencies.docx`, Section 3, to be
`preprocessing.ipynb`'s real save location. Skip it, and use the
self-check in Section 10 instead, if you're reviewing the logic without
Drive access.


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
summaries_train = process_and_save_all_windowed_files(split="train")
summaries_train

summaries_val = process_and_save_all_windowed_files(split="val")
summaries_val


[1/9] Loading N-CMAPSS_DS01-005.h5 (train) ...
    read 4,069,786 rows
    built 81,393 windows (window_length=50, stride=50)
    saved to /content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/windowed/N-CMAPSS_DS01-005_train_windowed.npz (780.7 MB)
[2/9] Loading N-CMAPSS_DS02-006.h5 (train) ...
    read 4,495,287 rows
    built 89,903 windows (window_length=50, stride=50)
    saved to /content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/windowed/N-CMAPSS_DS02-006_train_windowed.npz (860.4 MB)
[3/9] Loading N-CMAPSS_DS03-012.h5 (train) ...
    read 4,132,089 rows
    built 82,638 windows (window_length=50, stride=50)
    saved to /content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/windowed/N-CMAPSS_DS03-012_train_windowed.npz (793.5 MB)
[4/9] Loading N-CMAPSS_DS04.h5 (train) ...
    read 5,190,386 rows
    built 103,805 windows (window_length=50, stride=50)
    saved to /content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/windowed/N-CMAPSS_DS04_train

[{'filename': 'N-CMAPSS_DS01-005.h5',
  'n_rows': 836850,
  'n_windows': 16737,
  'output_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/windowed/N-CMAPSS_DS01-005_val_windowed.npz',
  'output_mb': 160.82188987731934},
 {'filename': 'N-CMAPSS_DS02-006.h5',
  'n_rows': 768160,
  'n_windows': 15363,
  'output_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/windowed/N-CMAPSS_DS02-006_val_windowed.npz',
  'output_mb': 146.90894603729248},
 {'filename': 'N-CMAPSS_DS03-012.h5',
  'n_rows': 1439188,
  'n_windows': 28783,
  'output_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/windowed/N-CMAPSS_DS03-012_val_windowed.npz',
  'output_mb': 275.69262981414795},
 {'filename': 'N-CMAPSS_DS04.h5',
  'n_rows': 1187066,
  'n_windows': 23741,
  'output_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/windowed/N-CMAPSS_DS04_val_windowed.npz',
  'output_mb': 229.86752319335938},
 {'filename': 'N-CMAPSS_DS05.h5',
  'n

## 10. Optional self-check (no Drive needed)

Builds two tiny synthetic "processed file" `.npz` files — matching
`loaders.ipynb`'s exact saved format, with two units of different
lengths inside one of them — runs them through
`process_and_save_all_windowed_files()`, reloads with
`load_windowed_file()`, and streams both back with `iter_windowed_files()`.

In [12]:
def _self_check():
    with tempfile.TemporaryDirectory() as data_dir, tempfile.TemporaryDirectory() as out_dir:
        rng = np.random.default_rng(42)
        # DS05 has two units (9 and 6 rows); DS04 has one unit (12 rows).
        # window_length=10 below, so DS05's units (9, 6) are BOTH too
        # short and should be skipped; DS04's 12 rows give exactly 1 window.
        specs = {
            "N-CMAPSS_DS05.h5": {"units": {1: 9, 2: 6}},
            "N-CMAPSS_DS04.h5": {"units": {1: 12}},
        }
        expected_windows_per_file = {}

        for fname, spec in specs.items():
            rows = list(spec["units"].items())
            n_rows = sum(n for _, n in rows)

            A = np.zeros((n_rows, 4))
            labels = np.zeros((n_rows, 5), dtype=np.int8)
            offset, n_expected = 0, 0
            for unit, n in rows:
                A[offset:offset + n, 0] = unit
                A[offset:offset + n, 1] = np.arange(1, n + 1)
                labels[offset + n - 1, 0] = 1  # a fault on the last cycle of every unit
                n_expected += (n // 10) if n >= 10 else 0  # window_length=10, stride=10 used below
                offset += n

            file_data = {
                "W": rng.random((n_rows, 4)), "X_s": rng.random((n_rows, 14)), "X_v": rng.random((n_rows, 14)),
                "A": A, "Y": rng.random((n_rows, 1)), "labels": labels,
            }
            np.savez_compressed(os.path.join(data_dir, fname.replace(".h5", "_train.npz")), **file_data)
            expected_windows_per_file[fname] = n_expected

        test_files = list(specs.keys())
        summaries = process_and_save_all_windowed_files(
            data_dir=data_dir, out_dir=out_dir, split="train", files=test_files,
            window_length=10, stride=10, label_rule="last")

        for s in summaries:
            assert s["n_windows"] == expected_windows_per_file[s["filename"]], (s["filename"], s["n_windows"])
            assert os.path.exists(s["output_path"])

        reloaded = load_windowed_file("N-CMAPSS_DS05.h5", out_dir=out_dir, split="train")
        assert reloaded["X"].shape[1:] == (28, 10)  # both DS05 units were too short -- 0 windows, but shape stays well-formed
        assert set(reloaded["unit_id"].tolist()) <= {1.0, 2.0}

        total = sum(data["X"].shape[0] for _, data in iter_windowed_files(out_dir=out_dir, split="train", files=test_files))
        assert total == sum(expected_windows_per_file.values())

    print("ALL SELF-CHECKS PASSED")


_self_check()

[1/2] Loading N-CMAPSS_DS05.h5 (train) ...
    read 15 rows
    [skip] unit 1.0: only 9 rows (< window_length=10), no windows produced
    [skip] unit 2.0: only 6 rows (< window_length=10), no windows produced
    built 0 windows (window_length=10, stride=10)
    saved to /tmp/tmpivvs65jc/N-CMAPSS_DS05_train_windowed.npz (0.0 MB)
[2/2] Loading N-CMAPSS_DS04.h5 (train) ...
    read 12 rows
    built 1 windows (window_length=10, stride=10)
    saved to /tmp/tmpivvs65jc/N-CMAPSS_DS04_train_windowed.npz (0.0 MB)
Done -- 2 files processed, 1 total windows saved to /tmp/tmpivvs65jc
ALL SELF-CHECKS PASSED


## 11. For learning: "last" vs "any" side by side (not a test)

A plain printout, not an assertion — built so the difference between
the two label rules from Section 2 can be **seen**, not just read about.
One fault flag is buried in the middle of the second window; watch how
each rule handles it differently.

In [13]:
def _demo_last_vs_any():
    n_rows, L, stride = 12, 4, 4
    X = np.zeros((n_rows, 1))  # values don't matter for this demo -- only the labels do
    labels = np.zeros((n_rows, 5), dtype=np.int8)
    labels[5, 2] = 1  # a hpc_fail flag buried in the MIDDLE of what becomes window index 1 (rows 4-7)

    _, labels_last, starts = _windows_for_one_unit(X, labels, window_length=L, stride=stride, label_rule="last")
    _, labels_any, _ = _windows_for_one_unit(X, labels, window_length=L, stride=stride, label_rule="any")

    print(f"{'window':>8} {'rows covered':>14} {'last-row label':>16} {'any-row label':>16}")
    for i, start in enumerate(starts):
        rows_str = f"{start}-{start + L - 1}"
        print(f"{i:>8} {rows_str:>14} {str(labels_last[i].tolist()):>16} {str(labels_any[i].tolist()):>16}")


_demo_last_vs_any()

  window   rows covered   last-row label    any-row label
       0            0-3  [0, 0, 0, 0, 0]  [0, 0, 0, 0, 0]
       1            4-7  [0, 0, 0, 0, 0]  [0, 0, 1, 0, 0]
       2           8-11  [0, 0, 0, 0, 0]  [0, 0, 0, 0, 0]


## What's next

With windowing built and verified against synthetic data -- correct
shapes, unit boundaries never crossed, both label rules and both
channel modes proven, short units skipped loudly -- and its input
location now confirmed (`data/preprocessed/`, per
`pipeline_sequence_and_dependencies.docx`, Section 3), the remaining gap
before a real trained model is:

1. **Run Section 9 for real**, once per split (`"train"` and `"val"`),
   to produce actual `data/windowed/*.npz` files from real N-CMAPSS data.
2. **`training/train.py`** -- the first place `MA1DCNN` (from
   `ma1dcnn.ipynb`) and this notebook's real windowed output finally
   meet, per the two-track diagram in `pipeline_sequence_and_dependencies.docx`.
